# Distance to Nearest Pharmacy from SAL Centroids
## Euclidean check + OSMnx network download

**Tess Vu**

Downloads walk and drive networks for Gauteng and KwaZulu-Natal and saves them as GraphML for downstream notebooks. Also computes a quick Euclidean-distance check (superseded by `sal_pharmacy_distance_k3.ipynb`).

CRS: EPSG:32735 (UTM 35S, meters) for distance work, graphs are native EPSG:4326.

- Outputs: `data/networks/network_{gauteng,kwazulu_natal}_{walk,drive}.graphml`
- Runtime: 26–88 min per graph on first download; cached graphs load from disk.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
from scipy.spatial import cKDTree
from src.crs import CRS_UTM35S, CRS_WGS84
from src.io import ensure_dir
from src.paths import NETWORKS, PHARMACIES_MASTER, POP_PRED_FINAL, SAL_W_WARD_DEDUP

print(f"OSMnx Version: {ox.__version__}")

OSMnx Version: 2.1.0


In [ ]:
ensure_dir(NETWORKS)
PROVINCES = ["Gauteng", "KwaZulu-Natal"]
print(f"Output Directory: {NETWORKS}")

In [ ]:
sal_geo = gpd.read_file(SAL_W_WARD_DEDUP)
print(f"SAL shapefile loaded: {sal_geo.shape[0]} features")
print(f"CRS: {sal_geo.crs}")

pop_est = pd.read_csv(POP_PRED_FINAL)
print(f"Population estimates loaded: {pop_est.shape[0]} rows")

Output Directory: c:\Users\Tess\Desktop\UPenn\UPenn_SS26\MUSA_8010-001_Practicum\south-africa-healthcare\notebooks\data\networks


In [ ]:
sal = sal_geo.merge(
    pop_est[["EA_CODE", "sal2023_est", "WardID"]],
    on="EA_CODE",
    how="left",
)

matched = sal["sal2023_est"].notna().sum()
print(f"JOIN RESULTS: {matched}/{sal.shape[0]} SALs matched ({matched/sal.shape[0]*100:.1f}%)")

sal = sal[sal["sal2023_est"].notna()].copy()
print(f"SALs retained after dropping unmatched: {sal.shape[0]}")

## SAL Centroids

Geometric centroids in EPSG:32735, lon/lat copies for OSMnx.

**Limitation:** geometric centroids can fall in uninhabited areas for
irregular SALs. Population-weighted centroids from building footprints
would be more defensible.

In [ ]:
sal_proj = sal.to_crs(CRS_UTM35S)
sal_proj["centroid_geom"] = sal_proj.geometry.centroid
sal_proj["centroid_x"] = sal_proj["centroid_geom"].x
sal_proj["centroid_y"] = sal_proj["centroid_geom"].y

# lon/lat centroids for OSMnx (computed in projected space, then reprojected)
centroids_wgs84 = sal_proj["centroid_geom"].to_crs(CRS_WGS84)
sal_proj["centroid_lat"] = centroids_wgs84.y.values
sal_proj["centroid_lng"] = centroids_wgs84.x.values

print(f"SAL CENTROIDS COMPUTED: {sal_proj.shape[0]} centroids")
print(f"Centroid X range: {sal_proj['centroid_x'].min():.0f} to {sal_proj['centroid_x'].max():.0f} m")
print(f"Centroid Y range: {sal_proj['centroid_y'].min():.0f} to {sal_proj['centroid_y'].max():.0f} m")

Population estimates loaded: 38380 rows
Columns: ['WardID', 'EA_CODE', 'sal2011_pop', 'ward2023_pop', 'EA_GTYPE', 'EA_TYPE', 'econ_status', 'houses2011', 'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other', 'area_km2', 'sal_dense', 'log_density', 'ward2011_sum', 'share2011', 'dasym_weight', 'sal2023_est', 'growth_rate']


In [ ]:
# Pharmacy master file: LAT/LNG are uppercase in the on-disk contract (EPSG:4326).
pharm_df = pd.read_csv(PHARMACIES_MASTER)
print(f"Pharmacies loaded: {pharm_df.shape[0]} rows")

assert "LAT" in pharm_df.columns, "Missing 'LAT' column in pharmacy file."
assert "LNG" in pharm_df.columns, "Missing 'LNG' column in pharmacy file."

valid_coords = pharm_df["LAT"].notna() & pharm_df["LNG"].notna()
print(f"Pharmacies with valid coordinates: {valid_coords.sum()}/{pharm_df.shape[0]}")
pharm_df = pharm_df[valid_coords].copy()

pharm_gdf = gpd.GeoDataFrame(
    pharm_df,
    geometry=gpd.points_from_xy(pharm_df["LNG"], pharm_df["LAT"]),
    crs=CRS_WGS84,
)
pharm_proj = pharm_gdf.to_crs(CRS_UTM35S)
pharm_proj["pharm_x"] = pharm_proj.geometry.x
pharm_proj["pharm_y"] = pharm_proj.geometry.y

print(f"PHARMACIES READY: {pharm_proj.shape[0]} geocoded locations")

JOIN RESULTS: 39177/39177 SALs matched (100.0%)
SALs retained after dropping unmatched: 39177


In [ ]:
print("BOUNDING BOX COMPARISON (WGS84)")
sal_bounds = sal.to_crs(CRS_WGS84).total_bounds
pharm_bounds = pharm_gdf.total_bounds
print(f"SALs: W={sal_bounds[0]:.3f}, S={sal_bounds[1]:.3f}, E={sal_bounds[2]:.3f}, N={sal_bounds[3]:.3f}")
print(f"Pharmacies: W={pharm_bounds[0]:.3f}, S={pharm_bounds[1]:.3f}, E={pharm_bounds[2]:.3f}, N={pharm_bounds[3]:.3f}")

## Euclidean Distance Check (KDTree, meters)

Straight-line lower bound. `sal_pharmacy_distance_k3.ipynb` supersedes these outputs.

## LOAD GEOCODED PHARMACIES
Load the fully geocoded pharmacy file and convert to a GeoDataFrame.

In [ ]:
pharm_coords = np.column_stack([pharm_proj["pharm_x"].values, pharm_proj["pharm_y"].values])
tree = cKDTree(pharm_coords)

sal_coords = np.column_stack([sal_proj["centroid_x"].values, sal_proj["centroid_y"].values])
distances_m, indices = tree.query(sal_coords, k=1)

sal_proj["euclidean_dist_m"] = distances_m
sal_proj["euclidean_dist_km"] = distances_m / 1000.0
sal_proj["nearest_pharm_idx"] = indices

print("EUCLIDEAN DISTANCE COMPUTED")
print(sal_proj["euclidean_dist_km"].describe().to_string())

Pharmacies loaded: 2241 rows
Columns: ['RECORD_ID', 'Y_NUMBER', 'PHARMACY_ID', 'PLACE_ID', 'NAME', 'STATUS', 'LICENCE_NUMBER', 'REGISTRATION_DATE', 'OWNER', 'INSPECTION', 'ADDRESS', 'CITY', 'PROVINCE', 'TELEPHONE', 'MATCHED_NAME', 'MATCHED_ADDRESS', 'LAT', 'LNG', 'TYPES', 'OPENING_HOURS', 'SOURCE', 'SPATIAL_CHECK']
Pharmacies with valid coordinates: 2241/2241
PHARMACIES READY: 2241 geocoded locations


In [ ]:
# Distance choropleth per province.
prov_col = next((c for c in ["PR_NAME", "PROVINCE", "Province", "province"]
                 if c in sal_proj.columns), None)
if prov_col is None:
    prov_col = "ALL"
    sal_proj["ALL"] = "All Provinces"
print(f"Province column: {prov_col}")

dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
cmap = mpl.cm.RdYlGn_r

for i, prov in enumerate(sal_proj[prov_col].unique()[:2]):
    subset = sal_proj[(sal_proj[prov_col] == prov) & sal_proj["euclidean_dist_km"].notna()]
    subset.plot(
        column="euclidean_dist_km",
        cmap=cmap,
        scheme="UserDefined",
        classification_kwds={"bins": dist_bins_km},
        legend=True,
        legend_kwds={"title": "Distance (km)", "loc": "lower right", "fontsize": 8},
        ax=axes[i],
        edgecolor="none",
        linewidth=0,
    )
    axes[i].set_title(f"{prov}: Euclidean Distance to Nearest Pharmacy", fontsize=12)
    axes[i].axis("off")

plt.suptitle("SAL Centroid Euclidean Distance to Nearest Pharmacy (EPSG:32735, km)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

BOUNDING BOX COMPARISON (WGS84)
SALs: W=27.156, S=-31.083, E=32.891, N=-25.110
Pharmacies: W=17.989, S=-34.091, E=32.757, N=-23.697


## Download OSMnx network graphs

Walk and drive networks per province, saved as GraphML (native EPSG:4326). Existing files load from disk, first-time downloads take 26–88 min per graph.

In [ ]:
ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.timeout = 600

place_queries = {
    "Gauteng": "Gauteng, South Africa",
    "KwaZulu-Natal": "KwaZulu-Natal, South Africa",
}
network_types = ["walk", "drive"]

graphs = {}
for prov_name, place_query in place_queries.items():
    for net_type in network_types:
        graph_key = f"{prov_name}_{net_type}"
        graphml_path = NETWORKS / f"network_{prov_name.lower().replace('-', '_')}_{net_type}.graphml"

        if graphml_path.exists():
            print(f"CACHED GRAPH: {graph_key}")
            G = ox.load_graphml(graphml_path)
        else:
            print(f"DOWNLOADING {net_type.upper()} NETWORK: {place_query}")
            start_time = time.time()
            G = ox.graph_from_place(place_query, network_type=net_type)
            print(f"Download complete in {time.time() - start_time:.1f} seconds.")

            if not all("length" in data for _, _, data in G.edges(data=True)):
                G = ox.distance.add_edge_lengths(G)

            ox.save_graphml(G, graphml_path)
            print(f"Graph saved to {graphml_path}")

        graphs[graph_key] = G
        print(f"  Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}\n")

print(f"Graph Keys: {list(graphs.keys())}")

EUCLIDEAN DISTANCE COMPUTED
count    39177.000000
mean         4.465366
std          6.953022
min          0.005177
25%          0.659558
50%          1.349065
75%          4.272367
max         44.852519


## Notes

- Geometric centroids may fall in uninhabited areas for irregular or large rural SALs. Population-weighted centroids from building footprints would reduce this bias.
- OSM coverage gaps in rural and informal settlement areas mean longer detours or unreachable centroids, hence the reachable-percentage diagnostics downstream.
- Drive networks are directed (one-way streets matter).
- The walk network includes mapped sidewalks and footpaths, but informal-settlement paths are often unmapped.